# PJM Load Forecasting Evaluation

This notebook displays the real-world electric load forecasting benchmark added to `qrc-engine`. It compares classical baselines and QRC backends on PJM East hourly demand using the same features, split, normalization, and metrics.

## Setup

The notebook reuses the helper functions from `experiments/load_forecasting.py`, so the notebook and script stay aligned.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from experiments.load_forecasting import (
    DATA_PATH,
    N_QRC_TEST,
    N_QRC_TRAIN,
    OUT_DIR,
    WASHOUT,
    evaluate,
    fit_classical_models,
    fit_qrc_models,
    load_dataset,
    print_table,
    save_context_plot,
    save_rmse_plot,
    save_time_domain_plot,
    scale_features,
)

plt.style.use('seaborn-v0_8-whitegrid')
DATA_PATH

## Load and Prepare Data

In [ ]:
X_train, y_train, X_test, y_test = load_dataset()
X_train_scaled, X_test_scaled = scale_features(X_train, X_test)

print(f'Train shape: {X_train_scaled.shape}')
print(f'Test shape:  {X_test_scaled.shape}')
print('Feature count:', X_train_scaled.shape[1])

## Classical Models on the Full 2018 Test Set

In [ ]:
classical_full = fit_classical_models(X_train_scaled, y_train, X_test_scaled)
full_results = [
    evaluate('Ridge (full data)', y_test, classical_full['Ridge']),
    evaluate('Random Forest (full data)', y_test, classical_full['Random Forest']),
    evaluate('MLP (full data)', y_test, classical_full['MLP']),
    evaluate('GAM-like (full data)', y_test, classical_full['GAM-like']),
]
print_table(full_results, 'Classical models - full training set')
full_results

## Fair 5,000 / 2,000 Window for Classical vs QRC

In [ ]:
X_qrc_train = X_train_scaled[-N_QRC_TRAIN:]
y_qrc_train = y_train[-N_QRC_TRAIN:]
X_qrc_test = X_test_scaled[:N_QRC_TEST]
y_qrc_test = y_test[:N_QRC_TEST]
y_aligned = y_qrc_test[WASHOUT:]

classical_sub = fit_classical_models(X_qrc_train, y_qrc_train, X_qrc_test)
aligned_classical = {name: prediction[WASHOUT:] for name, prediction in classical_sub.items()}

qrc_predictions, qrc_timings = fit_qrc_models(X_qrc_train, y_qrc_train, X_qrc_test)

sub_results = [
    evaluate('Ridge', y_aligned, aligned_classical['Ridge']),
    evaluate('Random Forest', y_aligned, aligned_classical['Random Forest']),
    evaluate('MLP', y_aligned, aligned_classical['MLP']),
    evaluate('GAM-like', y_aligned, aligned_classical['GAM-like']),
    evaluate('QRC Qiskit (ridge)', y_aligned, qrc_predictions['QRC Qiskit (ridge)']),
    evaluate('QRC Qiskit (RF readout)', y_aligned, qrc_predictions['QRC Qiskit (RF readout)']),
    evaluate('QRC Dynamiqs', y_aligned, qrc_predictions['QRC Dynamiqs']),
    evaluate('QRC Perceval Fock+FB', y_aligned, qrc_predictions['QRC Perceval Fock+FB']),
]

print_table(sub_results, f'Fair comparison - same {N_QRC_TRAIN:,}-sample window')
qrc_timings

## Inline Comparison Plots

In [ ]:
save_time_domain_plot(y_aligned, aligned_classical, qrc_predictions, sub_results)
save_rmse_plot(sub_results)
save_context_plot(full_results, sub_results)

## Notes

- Classical models currently win clearly on this real-world benchmark.
- Among the tested QRC variants here, Dynamiqs performed best on the fair subsample.
- The notebook stays aligned with the script by importing its helpers directly.